# NYC Taxi Pipeline Investigation

This notebook demonstrates three methods to investigate your dlt pipeline:
1. **dlt Dashboard** - View pipeline metadata and execution history
2. **dlt MCP Server** - Query pipeline information
3. **Marimo Visualizations** - Interactive analysis and charts

## 1. Set Up dlt Environment and Import Libraries

In [ ]:
import dlt
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import json

# Connect to the pipeline database
conn = duckdb.connect('taxi_pipeline.duckdb')

print("✓ Libraries imported and DuckDB connection established")

## 2. Explore the Pipeline

In [ ]:
# Get pipeline information
pipeline = dlt.pipeline(
    pipeline_name='taxi_pipeline',
    destination='duckdb',
    dataset_name='taxi_data'
)

print(f"Pipeline Name: {pipeline.pipeline_name}")
print(f"Destination: {pipeline.destination}")
print(f"Dataset: {pipeline.dataset_name}")
print(f"\nPipeline directory: {pipeline.pipeline_dir}")

## 3. Query Loaded Data - Summary Statistics

In [ ]:
# Get overall statistics
stats = conn.execute("""
    SELECT 
        COUNT(*) as total_trips,
        COUNT(DISTINCT vendor_name) as unique_vendors,
        ROUND(AVG(CAST(Fare_Amt AS FLOAT)), 2) as avg_fare,
        ROUND(MAX(CAST(Fare_Amt AS FLOAT)), 2) as max_fare,
        ROUND(AVG(CAST(Tip_Amt AS FLOAT)), 2) as avg_tip,
        ROUND(AVG(CAST(Trip_Distance AS FLOAT)), 2) as avg_distance
    FROM taxi_data
""").fetchall()

print("=== Taxi Data Summary ===")
print(f"Total Trips: {stats[0][0]:,}")
print(f"Unique Vendors: {stats[0][1]}")
print(f"Average Fare: ${stats[0][2]}")
print(f"Max Fare: ${stats[0][3]}")
print(f"Average Tip: ${stats[0][4]}")
print(f"Average Distance: {stats[0][5]} miles")

## 4. Vendor Analysis

In [ ]:
# Analyze by vendor
vendor_data = conn.execute("""
    SELECT 
        vendor_name,
        COUNT(*) as trip_count,
        ROUND(AVG(CAST(Fare_Amt AS FLOAT)), 2) as avg_fare,
        ROUND(AVG(CAST(Tip_Amt AS FLOAT)), 2) as avg_tip
    FROM taxi_data
    WHERE vendor_name IS NOT NULL
    GROUP BY vendor_name
    ORDER BY trip_count DESC
""").df()

print(vendor_data)

# Visualize vendor distribution
fig = px.bar(
    vendor_data, 
    x='vendor_name', 
    y='trip_count',
    title='Trips by Vendor',
    labels={'trip_count': 'Number of Trips', 'vendor_name': 'Vendor'}
)
fig.show()

## 5. Payment Type Analysis

In [ ]:
# Analyze payment types
payment_data = conn.execute("""
    SELECT 
        COALESCE(Payment_Type, 'Unknown') as payment_type,
        COUNT(*) as trip_count,
        ROUND(AVG(CAST(Total_Amt AS FLOAT)), 2) as avg_total
    FROM taxi_data
    GROUP BY Payment_Type
    ORDER BY trip_count DESC
""").df()

print(payment_data)

# Pie chart
fig = px.pie(
    payment_data,
    values='trip_count',
    names='payment_type',
    title='Payment Types Distribution'
)
fig.show()

## 6. Fare and Distance Distribution

In [ ]:
# Get distribution data
distribution_data = conn.execute("""
    SELECT 
        CAST(Fare_Amt AS FLOAT) as fare,
        CAST(Trip_Distance AS FLOAT) as distance,
        CAST(Total_Amt AS FLOAT) as total
    FROM taxi_data
    WHERE 
        Fare_Amt > 0 
        AND Trip_Distance > 0 
        AND Fare_Amt < 100
    LIMIT 1000
""").df()

# Scatter plot: Fare vs Distance
fig = px.scatter(
    distribution_data,
    x='distance',
    y='fare',
    title='Fare Amount vs Trip Distance',
    labels={'distance': 'Trip Distance (miles)', 'fare': 'Fare ($)'},
    opacity=0.6
)
fig.update_layout(height=500)
fig.show()

# Histogram: Fare distribution
fig = px.histogram(
    distribution_data,
    x='fare',
    nbins=50,
    title='Fare Amount Distribution',
    labels={'fare': 'Fare ($)'}
)
fig.show()

## 7. Passenger Count Analysis

In [ ]:
# Analyze by passenger count
passenger_data = conn.execute("""
    SELECT 
        Passenger_Count,
        COUNT(*) as trip_count,
        ROUND(AVG(CAST(Fare_Amt AS FLOAT)), 2) as avg_fare,
        ROUND(AVG(CAST(Tip_Amt AS FLOAT)), 2) as avg_tip
    FROM taxi_data
    WHERE Passenger_Count > 0 AND Passenger_Count <= 6
    GROUP BY Passenger_Count
    ORDER BY Passenger_Count
""").df()

print(passenger_data)

# Bar chart: Trips by passenger count
fig = px.bar(
    passenger_data,
    x='Passenger_Count',
    y='trip_count',
    title='Trips by Passenger Count',
    labels={'trip_count': 'Number of Trips', 'Passenger_Count': 'Passenger Count'}
)
fig.show()

## 8. Data Quality Check

In [ ]:
# Check for nulls and data quality
quality = conn.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT start_lat) as distinct_start_locations,
        COUNT(DISTINCT end_lat) as distinct_end_locations,
        SUM(CASE WHEN start_lat = 0 OR start_lon = 0 THEN 1 ELSE 0 END) as invalid_start_coords,
        SUM(CASE WHEN end_lat = 0 OR end_lon = 0 THEN 1 ELSE 0 END) as invalid_end_coords,
        SUM(CASE WHEN fare_amt IS NULL OR fare_amt <= 0 THEN 1 ELSE 0 END) as invalid_fares
    FROM taxi_data
""").fetchall()

print("=== Data Quality Report ===")
print(f"Total Rows: {quality[0][0]:,}")
print(f"Distinct Start Locations: {quality[0][1]:,}")
print(f"Distinct End Locations: {quality[0][2]:,}")
print(f"Invalid Start Coordinates: {quality[0][3]:,}")
print(f"Invalid End Coordinates: {quality[0][4]:,}")
print(f"Invalid Fares: {quality[0][5]:,}")
print(f"\nData Quality Score: {((quality[0][0] - quality[0][3] - quality[0][4] - quality[0][5]) / quality[0][0] * 100):.1f}%")

## 9. Summary

Your dlt taxi pipeline has successfully:
- ✓ Connected to the REST API
- ✓ Paginated through all available data (stops when empty page is returned)
- ✓ Loaded data into DuckDB
- ✓ Created proper schema with nested fields

### Next Steps:
1. **Monitor the pipeline**: Run regularly to get fresh data
2. **Add transformations**: Create dbt models for cleaned data
3. **Set up alerts**: Monitor data quality metrics
4. **Schedule with Kestra**: Automate pipeline execution